In [3]:
import pandas as pd
import numpy as np
import re

In [4]:
df = pd.read_csv("train.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns)
df.head()

Shape: (159571, 8)

Columns: Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [8]:
df.isnull().sum()

id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

In [9]:
label_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

df[label_cols].sum()

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

In [5]:
def assign_label(row):
    if (row['severe_toxic'] == 1) or (row['threat'] == 1) or (row['identity_hate'] == 1):
        return "severe_abuse"
    elif (row['toxic'] == 1) or (row['insult'] == 1) or (row['obscene'] == 1):
        return "toxic"
    else:
        return "clean"

df['final_label'] = df.apply(assign_label, axis=1)

In [11]:
df['final_label'].value_counts()

final_label
clean           143346
toxic            13238
severe_abuse      2987
Name: count, dtype: int64

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

df['cleaned_text'] = df['comment_text'].apply(clean_text)

In [13]:
df[['comment_text','cleaned_text','final_label']].head()

,comment_text,cleaned_text,final_label
0,Explanation\nWhy the edits made under my usern...,explanation why the edits made under my userna...,clean
1,D'aww! He matches this background colour I'm s...,daww he matches this background colour im seem...,clean
2,"Hey man, I'm really not trying to edit war. It...",hey man im really not trying to edit war its j...,clean
3,"""\nMore\nI can't make any real suggestions on ...",more i cant make any real suggestions on impro...,clean
4,"You, sir, are my hero. Any chance you remember...",you sir are my hero any chance you remember wh...,clean


## EDA

**CLASS DISTRIBUTION**

In [14]:
df['final_label'].value_counts(normalize=True)

final_label
clean           0.898321
toxic           0.082960
severe_abuse    0.018719
Name: proportion, dtype: float64

**COMMENT LENGTH ANALYSIS**

In [15]:
df['comment_length'] = df['cleaned_text'].apply(len)

df.groupby('final_label')['comment_length'].mean()

final_label
clean           376.948698
severe_abuse    336.417141
toxic           268.342499
Name: comment_length, dtype: float64

**MOST COMMON WORDS**

In [16]:
from collections import Counter

all_words = " ".join(df['cleaned_text']).split()
Counter(all_words).most_common(20)

[('the', 495480),
 ('to', 296851),
 ('of', 224025),
 ('and', 222368),
 ('a', 214910),
 ('you', 204560),
 ('i', 200655),
 ('is', 175960),
 ('that', 154301),
 ('in', 144188),
 ('it', 129646),
 ('for', 102445),
 ('this', 97083),
 ('not', 93338),
 ('on', 89447),
 ('be', 83327),
 ('as', 77249),
 ('have', 72173),
 ('are', 71876),
 ('your', 63253)]

## PREPARE FOR ML

In [7]:
X = df['cleaned_text']
y = df['final_label']

## TRAIN-TEST SPLIT

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## TF-IDF VECTORIZATION

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## TRAIN MODEL (LOGISTIC REGRESSION)

In [21]:
from sklearn.linear_model import LogisticRegression

ml_model = LogisticRegression(
    max_iter=300,
    class_weight='balanced',
    C=2
)

ml_model.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,2
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,300
,multi_class,'deprecated'


In [22]:
probs = ml_model.predict_proba(X_test_vec)

threshold = 0.6

y_pred_custom = []

for prob in probs:
    max_prob = np.max(prob)
    label = ml_model.classes_[np.argmax(prob)]
    
    if max_prob < threshold:
        y_pred_custom.append("clean")
    else:
        y_pred_custom.append(label)

# PREDICTION

In [23]:
y_pred = ml_model.predict(X_test_vec)

## EVALUATION

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       clean       0.98      0.93      0.95     28670
severe_abuse       0.40      0.63      0.49       597
       toxic       0.46      0.68      0.55      2648

    accuracy                           0.90     31915
   macro avg       0.61      0.75      0.66     31915
weighted avg       0.93      0.90      0.91     31915

[[26535   167  1968]
 [   34   376   187]
 [  452   394  1802]]


In [25]:
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

       clean       0.97      0.95      0.96     28670
severe_abuse       0.46      0.56      0.51       597
       toxic       0.52      0.59      0.55      2648

    accuracy                           0.91     31915
   macro avg       0.65      0.70      0.67     31915
weighted avg       0.92      0.91      0.92     31915



In [11]:
import os
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents="Classify this comment: You are stupid. Answer only: clean, toxic, or severe_abuse."
)

print(response.text)

toxic


In [12]:
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def classify_comment_llm(comment):
    prompt = f"""
    You are a content moderation system.

    Classify the comment into ONE category:
    clean, toxic, severe_abuse

    Definitions:
    clean = normal or harmless
    toxic = rude, insulting, offensive
    severe_abuse = threats, hate speech, extreme harassment

    Comment: "{comment}"

    Output ONLY one word: clean, toxic, or severe_abuse.
    """

    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt
    )

    return response.text.strip().lower()

In [13]:
print(classify_comment_llm("I love this video"))
print(classify_comment_llm("You are so dumb"))
print(classify_comment_llm("I will kill you"))

clean
toxic
severe_abuse


In [14]:
def clean_llm_output(text):
    text = text.lower()
    if "severe" in text:
        return "severe_abuse"
    elif "toxic" in text:
        return "toxic"
    else:
        return "clean"

In [15]:
sample_df = df.groupby('final_label').apply(lambda x: x.sample(5)).reset_index(drop=True)

sample_df['llm_pred'] = sample_df['cleaned_text'].apply(classify_comment_llm)
sample_df['llm_pred'] = sample_df['llm_pred'].apply(clean_llm_output)

sample_df[['cleaned_text','final_label','llm_pred']]

C:\Users\Madhurima\AppData\Local\Temp\ipykernel_17700\2131606043.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = df.groupby('final_label').apply(lambda x: x.sample(5)).reset_index(drop=True)


,cleaned_text,final_label,llm_pred
0,im very sorry that i will be unable to help wi...,clean,clean
1,august utc i certainly do not have a problem w...,clean,clean
2,the name of the person to hold this office was...,clean,clean
3,hes an english professor he is armenian its hi...,clean,clean
4,this is my original account that dates back to...,clean,clean
5,ya muthafuckers blocked you guys are dumbasses...,severe_abuse,toxic
6,amerikkkans are fucking gay usa sucks balls th...,severe_abuse,severe_abuse
7,why dont you go back to turkey if it is so fan...,severe_abuse,severe_abuse
8,fuck you all you losers who ahve nothing bette...,severe_abuse,toxic
9,yeah where it says seriouslygay thats implying...,severe_abuse,toxic


In [16]:
from sklearn.metrics import classification_report

print(classification_report(sample_df['final_label'], sample_df['llm_pred']))

              precision    recall  f1-score   support

       clean       1.00      1.00      1.00         5
severe_abuse       0.67      0.40      0.50         5
       toxic       0.57      0.80      0.67         5

    accuracy                           0.73        15
   macro avg       0.75      0.73      0.72        15
weighted avg       0.75      0.73      0.72        15



In [17]:
def classify_comment_llm_fewshot(comment):
    prompt = f"""
    You are a content moderation system.

    Examples:
    Comment: "I love this"
    Label: clean

    Comment: "You are stupid"
    Label: toxic

    Comment: "I will kill you"
    Label: severe_abuse

    Now classify:

    Comment: "{comment}"
    Label:
    """

    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt
    )

    return response.text.strip().lower()

In [18]:
def clean_llm_output(text):
    text = text.lower().strip()
    
    if "severe" in text:
        return "severe_abuse"
    elif "toxic" in text:
        return "toxic"
    else:
        return "clean"

In [19]:
sample_df['llm_fewshot'] = sample_df['comment_text'].apply(classify_comment_llm_fewshot)
sample_df['llm_fewshot'] = sample_df['llm_fewshot'].apply(clean_llm_output)

In [26]:
sample_df['ml_pred'] = ml_model.predict(
    vectorizer.transform(sample_df['cleaned_text'])
)

In [27]:
sample_df[['comment_text','final_label','ml_pred','llm_fewshot']]

,comment_text,final_label,ml_pred,llm_fewshot
0,"I'm very sorry, , that I will be unable to hel...",clean,clean,clean
1,""", 11 August 2006 (UTC)\n\nI certainly do not ...",clean,clean,clean
2,"""The name of the person to hold this office wa...",clean,clean,clean
3,He's an English professor. He is Armenian. It'...,clean,clean,clean
4,This is my original account that dates back to...,clean,clean,clean
5,Ya muthafuckers \n\nblocked You guys are dumba...,severe_abuse,severe_abuse,toxic
6,... AmeriKKKans are fucking gay ... U.S.A. suc...,severe_abuse,severe_abuse,severe_abuse
7,Why don't you go back to Turkey if it is so fa...,severe_abuse,severe_abuse,severe_abuse
8,FUCK YOU ALL YOU LOSERS WHO AHVE NOTHING BETTE...,severe_abuse,toxic,severe_abuse
9,Yeah where it says seriously...gay thats imply...,severe_abuse,severe_abuse,toxic


In [28]:
from sklearn.metrics import classification_report

print(classification_report(sample_df['final_label'], sample_df['llm_fewshot']))

              precision    recall  f1-score   support

       clean       0.83      1.00      0.91         5
severe_abuse       1.00      0.60      0.75         5
       toxic       0.67      0.80      0.73         5

    accuracy                           0.80        15
   macro avg       0.83      0.80      0.80        15
weighted avg       0.83      0.80      0.80        15

